# Household power forecasting

Elton Lewis and Vignesh Rao

We compare MLP, 1D-CNN and LSTM using identical chronological splits and three random seeds. All results below are computed on Kaggle. Code and documentation preparation used OpenAI Codex assistance. See the repository report for scope and limitations.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
ROOT = Path("/kaggle/working/energy_forecasting")
ROOT.mkdir(exist_ok=True)
os.chdir(ROOT)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
sources = {'src/__init__.py': '# Deep Learning Project - Household Power Consumption Forecasting\n', 'src/compare_all.py': 'import numpy as np\nimport pandas as pd\nimport sys\nfrom pathlib import Path\n\n# Ensure project root is in sys.path\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\nif str(PROJECT_ROOT) not in sys.path:\n    sys.path.insert(0, str(PROJECT_ROOT))\n\nfrom src.evaluate import calculate_metrics, calculate_hourly_mae, plot_sample_forecast, plot_horizon_error, generate_comparison_table\n\ndef main():\n    root = Path(__file__).resolve().parent.parent\n    saved_dir = root / "saved_models"\n    plots_dir = root / "plots"\n    plots_dir.mkdir(parents=True, exist_ok=True)\n\n    trues_path = saved_dir / "trues_test.npy"\n    if not trues_path.exists():\n        print(f"Ground truth file not found at {trues_path}. Please train at least one model first.")\n        return\n\n    y_true = np.load(trues_path)\n    models = ["mlp", "cnn1d", "lstm"]\n    model_labels = {"mlp": "MLP (Baseline)", "cnn1d": "1D-CNN", "lstm": "LSTM"}\n\n    results = {}\n    preds_dict = {}\n    hourly_mae_dict = {}\n\n    for m in models:\n        pred_path = saved_dir / f"preds_{m}.npy"\n        if pred_path.exists():\n            y_pred = np.load(pred_path)\n            label = model_labels[m]\n            preds_dict[label] = y_pred\n            results[label] = calculate_metrics(y_true, y_pred)\n            hourly_mae_dict[label] = calculate_hourly_mae(y_true, y_pred)\n            print(f"Loaded predictions for: {label}")\n        else:\n            print(f"Note: Predictions for {m.upper()} not found yet (run `python src/train.py --model {m}`).")\n\n    if not results:\n        print("No model predictions found to compare.")\n        return\n\n    # Comparison Table\n    df_results = generate_comparison_table(results)\n    print("\\n" + "=" * 65)\n    print("                OVERALL MODEL COMPARISON (TEST SET)")\n    print("=" * 65)\n    print(df_results.to_markdown())\n    print("=" * 65)\n\n    # Save to CSV\n    table_csv = plots_dir / "model_comparison_metrics.csv"\n    df_results.to_csv(table_csv)\n    print(f"\\nSaved metrics summary to: {table_csv}")\n\n    # Generate multi-sample 24h forecast overlay\n    forecast_plot_path = plots_dir / "forecast_comparison_24h.png"\n    plot_sample_forecast(y_true, preds_dict, sample_indices=[12, 120, 300], save_path=forecast_plot_path)\n\n    # Generate horizon error plot\n    horizon_plot_path = plots_dir / "horizon_mae_error.png"\n    plot_horizon_error(hourly_mae_dict, save_path=horizon_plot_path)\n\nif __name__ == "__main__":\n    main()\n', 'src/data_loader.py': 'import numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import Dataset, DataLoader\nfrom sklearn.preprocessing import MinMaxScaler\nfrom pathlib import Path\n\nclass PowerConsumptionDataset(Dataset):\n    """PyTorch Dataset for multi-step time-series forecasting."""\n    def __init__(self, X: np.ndarray, y: np.ndarray):\n        self.X = torch.tensor(X, dtype=torch.float32)\n        self.y = torch.tensor(y, dtype=torch.float32)\n\n    def __len__(self):\n        return len(self.X)\n\n    def __getitem__(self, idx):\n        return self.X[idx], self.y[idx]\n\ndef create_sliding_windows(data_features: np.ndarray, target_values: np.ndarray, input_len: int = 168, output_len: int = 24):\n    """\n    Creates rolling window sequences.\n    \n    Args:\n        data_features: array of shape (T, num_features)\n        target_values: array of shape (T, 1) or (T,) - target column to predict\n        input_len: number of past timesteps (default: 168 hours = 7 days)\n        output_len: number of future timesteps to forecast (default: 24 hours)\n    \n    Returns:\n        X: array of shape (N, input_len, num_features)\n        y: array of shape (N, output_len)\n    """\n    X_list, y_list = [], []\n    total_len = len(data_features)\n    for i in range(total_len - input_len - output_len + 1):\n        X_list.append(data_features[i : i + input_len])\n        y_list.append(target_values[i + input_len : i + input_len + output_len].flatten())\n    \n    return np.array(X_list), np.array(y_list)\n\ndef load_and_split_data(\n    csv_path: str = None,\n    input_len: int = 168,\n    output_len: int = 24,\n    train_ratio: float = 0.70,\n    val_ratio: float = 0.15,\n    batch_size: int = 64\n):\n    """\n    Loads hourly power consumption data, performs chronological split (no leakage),\n    fits MinMaxScaler ONLY on the training split, creates sliding windows,\n    and returns PyTorch DataLoaders along with the target scaler.\n    """\n    if csv_path is None:\n        csv_path = Path(__file__).resolve().parent.parent / "data" / "processed" / "household_power_hourly.csv"\n    \n    df = pd.read_csv(csv_path, index_col="datetime", parse_dates=True)\n\n    feature_cols = [\n        "Global_active_power", "Global_reactive_power", "Voltage",\n        "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3",\n        "hour_sin", "hour_cos", "day_sin", "day_cos", "month_sin", "month_cos", "is_weekend"\n    ]\n    target_col = "Global_active_power"\n\n    data = df[feature_cols].values\n    target = df[[target_col]].values\n\n    # Chronological Split (Strictly preserve time order)\n    n = len(df)\n    train_end = int(n * train_ratio)\n    val_end = int(n * (train_ratio + val_ratio))\n\n    train_data = data[:train_end]\n    val_data = data[train_end:val_end]\n    test_data = data[val_end:]\n\n    train_target = target[:train_end]\n    val_target = target[train_end:val_end]\n    test_target = target[val_end:]\n\n    # Normalization (Fit scaler on train set ONLY to prevent data leakage)\n    feature_scaler = MinMaxScaler(feature_range=(0, 1))\n    target_scaler = MinMaxScaler(feature_range=(0, 1))\n\n    train_data_scaled = feature_scaler.fit_transform(train_data)\n    val_data_scaled = feature_scaler.transform(val_data)\n    test_data_scaled = feature_scaler.transform(test_data)\n\n    train_target_scaled = target_scaler.fit_transform(train_target)\n    val_target_scaled = target_scaler.transform(val_target)\n    test_target_scaled = target_scaler.transform(test_target)\n\n    # Sliding Windows\n    X_train, y_train = create_sliding_windows(train_data_scaled, train_target_scaled, input_len, output_len)\n    X_val, y_val = create_sliding_windows(val_data_scaled, val_target_scaled, input_len, output_len)\n    X_test, y_test = create_sliding_windows(test_data_scaled, test_target_scaled, input_len, output_len)\n\n    # PyTorch DataLoaders\n    train_dataset = PowerConsumptionDataset(X_train, y_train)\n    val_dataset = PowerConsumptionDataset(X_val, y_val)\n    test_dataset = PowerConsumptionDataset(X_test, y_test)\n\n    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)\n    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)\n    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)\n\n    metadata = {\n        "num_features": len(feature_cols),\n        "feature_cols": feature_cols,\n        "target_col": target_col,\n        "input_len": input_len,\n        "output_len": output_len,\n        "X_train_shape": X_train.shape,\n        "X_val_shape": X_val.shape,\n        "X_test_shape": X_test.shape,\n    }\n\n    return train_loader, val_loader, test_loader, target_scaler, metadata\n', 'src/download_and_preprocess.py': 'import os\nimport zipfile\nimport urllib.request\nimport numpy as np\nimport pandas as pd\nfrom pathlib import Path\n\nUCI_URL = "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\nRAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"\nPROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"\nRAW_ZIP_PATH = RAW_DATA_DIR / "household_power_consumption.zip"\nRAW_TXT_PATH = RAW_DATA_DIR / "household_power_consumption.txt"\nOUTPUT_CSV_PATH = PROCESSED_DATA_DIR / "household_power_hourly.csv"\n\ndef download_and_extract():\n    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)\n    PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)\n    (PROJECT_ROOT / "notebooks").mkdir(parents=True, exist_ok=True)\n    (PROJECT_ROOT / "src" / "models").mkdir(parents=True, exist_ok=True)\n\n    if not RAW_TXT_PATH.exists():\n        if not RAW_ZIP_PATH.exists():\n            print(f"[1/4] Downloading dataset from {UCI_URL}...")\n            headers = {"User-Agent": "Mozilla/5.0"}\n            req = urllib.request.Request(UCI_URL, headers=headers)\n            with urllib.request.urlopen(req) as response, open(RAW_ZIP_PATH, "wb") as out_file:\n                total_size = response.headers.get("Content-Length")\n                total_size = int(total_size) if total_size else None\n                downloaded = 0\n                block_size = 1024 * 1024  # 1MB\n                while True:\n                    buffer = response.read(block_size)\n                    if not buffer:\n                        break\n                    downloaded += len(buffer)\n                    out_file.write(buffer)\n                    if total_size:\n                        percent = downloaded / total_size * 100\n                        print(f"  Downloaded: {downloaded / (1024*1024):.1f} MB / {total_size / (1024*1024):.1f} MB ({percent:.1f}%)", end="\\r")\n                    else:\n                        print(f"  Downloaded: {downloaded / (1024*1024):.1f} MB", end="\\r")\n            print(f"\\nDownload complete: {RAW_ZIP_PATH}")\n        \n        print("[2/4] Extracting zip file...")\n        with zipfile.ZipFile(RAW_ZIP_PATH, "r") as zip_ref:\n            zip_ref.extractall(RAW_DATA_DIR)\n        print(f"Extracted to: {RAW_TXT_PATH}")\n    else:\n        print(f"Raw file already exists at {RAW_TXT_PATH}. Skipping download.")\n\ndef preprocess_and_resample():\n    print("[3/4] Reading raw minute-level data (2+ million rows)...")\n    # Columns: Date;Time;Global_active_power;Global_reactive_power;Voltage;Global_intensity;Sub_metering_1;Sub_metering_2;Sub_metering_3\n    df = pd.read_csv(\n        RAW_TXT_PATH,\n        sep=";",\n        na_values=["?", ""],\n        low_memory=False\n    )\n    print(f"  Raw rows read: {len(df):,}")\n\n    # Combine Date and Time\n    print("  Parsing datetime and handling missing values...")\n    df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], format="%d/%m/%Y %H:%M:%S")\n    df.drop(columns=["Date", "Time"], inplace=True)\n    df.set_index("datetime", inplace=True)\n\n    numeric_cols = [\n        "Global_active_power", "Global_reactive_power", "Voltage",\n        "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"\n    ]\n    for col in numeric_cols:\n        df[col] = pd.to_numeric(df[col], errors="coerce")\n\n    # Linear interpolation for small missing bursts at the minute level\n    df[numeric_cols] = df[numeric_cols].interpolate(method="time").ffill().bfill()\n\n    print("[4/4] Resampling from 1-minute to 1-hour intervals...")\n    # Resample to hourly mean\n    df_hourly = df.resample("1h").mean()\n\n    # In case of any remaining gaps in hourly timestamps\n    df_hourly = df_hourly.interpolate(method="time").ffill().bfill()\n\n    # Engineer cyclical temporal features\n    print("  Adding temporal features (hour, day of week, month, cyclical sin/cos)...")\n    df_hourly["hour"] = df_hourly.index.hour\n    df_hourly["dayofweek"] = df_hourly.index.dayofweek\n    df_hourly["month"] = df_hourly.index.month\n    df_hourly["is_weekend"] = (df_hourly["dayofweek"] >= 5).astype(int)\n\n    # Cyclical sin / cos encodings\n    df_hourly["hour_sin"] = np.sin(2 * np.pi * df_hourly["hour"] / 24.0)\n    df_hourly["hour_cos"] = np.cos(2 * np.pi * df_hourly["hour"] / 24.0)\n    df_hourly["day_sin"] = np.sin(2 * np.pi * df_hourly["dayofweek"] / 7.0)\n    df_hourly["day_cos"] = np.cos(2 * np.pi * df_hourly["dayofweek"] / 7.0)\n    df_hourly["month_sin"] = np.sin(2 * np.pi * (df_hourly["month"] - 1) / 12.0)\n    df_hourly["month_cos"] = np.cos(2 * np.pi * (df_hourly["month"] - 1) / 12.0)\n\n    # Save to CSV\n    df_hourly.to_csv(OUTPUT_CSV_PATH)\n    file_size_mb = OUTPUT_CSV_PATH.stat().st_size / (1024 * 1024)\n    print(f"\\nSuccessfully generated: {OUTPUT_CSV_PATH}")\n    print(f"Hourly dataset shape: {df_hourly.shape}")\n    print(f"File size: {file_size_mb:.2f} MB")\n    print("\\n--- Summary Statistics (Global_active_power) ---")\n    print(df_hourly["Global_active_power"].describe())\n    print("\\nFirst 3 rows:")\n    print(df_hourly.head(3))\n\nif __name__ == "__main__":\n    download_and_extract()\n    preprocess_and_resample()\n', 'src/evaluate.py': 'import numpy as np\nimport matplotlib.pyplot as plt\nimport pandas as pd\nfrom sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score\nfrom pathlib import Path\n\ndef calculate_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:\n    """\n    Computes standard regression metrics across the 24-hour horizon.\n    Assumes y_true and y_pred are in original kW units.\n    """\n    # Flatten across batch and time for overall metrics\n    y_true_flat = y_true.flatten()\n    y_pred_flat = y_pred.flatten()\n\n    mae = mean_absolute_error(y_true_flat, y_pred_flat)\n    mse = mean_squared_error(y_true_flat, y_pred_flat)\n    rmse = np.sqrt(mse)\n    r2 = r2_score(y_true_flat, y_pred_flat)\n\n    # Avoid division by zero in MAPE (epsilon = 1e-4)\n    epsilon = 1e-4\n    mape = np.mean(np.abs((y_true_flat - y_pred_flat) / np.clip(y_true_flat, epsilon, None))) * 100\n\n    return {\n        "MAE (kW)": float(mae),\n        "RMSE (kW)": float(rmse),\n        "MAPE (%)": float(mape),\n        "R2 Score": float(r2)\n    }\n\ndef calculate_hourly_mae(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:\n    """Calculates MAE for each horizon step (+1h to +24h)."""\n    return np.mean(np.abs(y_true - y_pred), axis=0)\n\ndef plot_sample_forecast(\n    y_true: np.ndarray,\n    predictions_dict: dict,\n    sample_indices: list = [0, 50, 100],\n    save_path: str = None\n):\n    """\n    Plots multi-step 24-hour forecast overlays comparing models against ground truth.\n    \n    predictions_dict: {"LSTM": y_pred_lstm, "1D-CNN": y_pred_cnn, "MLP": y_pred_mlp}\n    """\n    num_samples = len(sample_indices)\n    fig, axes = plt.subplots(num_samples, 1, figsize=(12, 4 * num_samples), sharex=True)\n    if num_samples == 1:\n        axes = [axes]\n\n    hours = np.arange(1, y_true.shape[1] + 1)\n\n    for i, idx in enumerate(sample_indices):\n        ax = axes[i]\n        ax.plot(hours, y_true[idx], label="Ground Truth (Actual)", color="black", linewidth=2.5, marker="o")\n        \n        for model_name, preds in predictions_dict.items():\n            ax.plot(hours, preds[idx], label=f"Predicted ({model_name})", linestyle="--", linewidth=1.8)\n\n        ax.set_title(f"24-Hour Ahead Forecast Sample #{idx}", fontsize=12, fontweight="bold")\n        ax.set_ylabel("Active Power (kW)", fontsize=10)\n        ax.grid(True, linestyle=":", alpha=0.6)\n        ax.legend(loc="upper right")\n\n    axes[-1].set_xlabel("Forecast Horizon (Hours Ahead)", fontsize=11)\n    plt.tight_layout()\n\n    if save_path:\n        Path(save_path).parent.mkdir(parents=True, exist_ok=True)\n        plt.savefig(save_path, dpi=300)\n        print(f"Saved forecast comparison plot to: {save_path}")\n    plt.close()\n\ndef plot_horizon_error(hourly_mae_dict: dict, save_path: str = None):\n    """\n    Plots MAE degradation across forecast horizon (+1h to +24h) for all models.\n    """\n    plt.figure(figsize=(10, 5))\n    for model_name, errors in hourly_mae_dict.items():\n        hours = np.arange(1, len(errors) + 1)\n        plt.plot(hours, errors, marker="o", label=model_name, linewidth=2)\n\n    plt.title("Forecast Error (MAE) Across 24-Hour Horizon", fontsize=13, fontweight="bold")\n    plt.xlabel("Forecast Step (Hours Ahead)", fontsize=11)\n    plt.ylabel("Mean Absolute Error (kW)", fontsize=11)\n    plt.xticks(np.arange(1, 25))\n    plt.grid(True, linestyle=":", alpha=0.6)\n    plt.legend()\n    plt.tight_layout()\n\n    if save_path:\n        Path(save_path).parent.mkdir(parents=True, exist_ok=True)\n        plt.savefig(save_path, dpi=300)\n        print(f"Saved horizon error plot to: {save_path}")\n    plt.close()\n\ndef generate_comparison_table(results_dict: dict) -> pd.DataFrame:\n    """\n    Creates a summary comparison DataFrame for all models.\n    results_dict format: {"Model Name": {"MAE (kW)": 0.35, "RMSE (kW)": 0.50, ...}}\n    """\n    df = pd.DataFrame(results_dict).T\n    return df\n', 'src/models/cnn1d.py': 'import torch\nimport torch.nn as nn\n\nclass CNN1DForecaster(nn.Module):\n    """\n    1D-Convolutional Neural Network to extract local temporal features across time.\n    Expects input shape: (batch_size, seq_len, num_features).\n    """\n    def __init__(\n        self,\n        num_features: int = 14,\n        filters: list = [32, 64, 128],\n        kernel_size: int = 3,\n        output_len: int = 24,\n        dropout: float = 0.2\n    ):\n        super(CNN1DForecaster, self).__init__()\n\n        # Conv1d expects (batch_size, in_channels, seq_len)\n        layers = []\n        in_ch = num_features\n        for out_ch in filters:\n            layers.append(nn.Conv1d(in_channels=in_ch, out_channels=out_ch, kernel_size=kernel_size, padding=kernel_size//2))\n            layers.append(nn.BatchNorm1d(out_ch))\n            layers.append(nn.ReLU())\n            layers.append(nn.MaxPool1d(kernel_size=2))\n            layers.append(nn.Dropout(dropout))\n            in_ch = out_ch\n\n        self.conv_net = nn.Sequential(*layers)\n        self.gap = nn.AdaptiveAvgPool1d(1)\n        self.fc = nn.Sequential(\n            nn.Linear(filters[-1], 64),\n            nn.ReLU(),\n            nn.Dropout(dropout),\n            nn.Linear(64, output_len)\n        )\n\n    def forward(self, x):\n        # x shape: (batch_size, seq_len, num_features)\n        # Permute to (batch_size, num_features, seq_len) for Conv1d\n        x = x.permute(0, 2, 1)\n        features = self.conv_net(x)\n        pooled = self.gap(features).squeeze(-1)\n        return self.fc(pooled)\n\nif __name__ == "__main__":\n    model = CNN1DForecaster()\n    dummy = torch.randn(32, 168, 14)\n    out = model(dummy)\n    print("1D-CNN Output shape:", out.shape)\n    print(f"Total trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")\n', 'src/models/lstm.py': 'import torch\nimport torch.nn as nn\n\nclass LSTMForecaster(nn.Module):\n    """\n    LSTM-based architecture for multi-step multivariate time-series forecasting.\n    \n    Args:\n        num_features: Number of input features at each timestep (e.g. 14).\n        hidden_dim: Number of hidden units in each LSTM layer.\n        num_layers: Number of stacked LSTM layers.\n        output_len: Forecast horizon (e.g. 24 hours).\n        dropout: Dropout rate between LSTM layers.\n    """\n    def __init__(\n        self,\n        num_features: int = 14,\n        hidden_dim: int = 64,\n        num_layers: int = 2,\n        output_len: int = 24,\n        dropout: float = 0.2\n    ):\n        super(LSTMForecaster, self).__init__()\n        self.num_layers = num_layers\n        self.hidden_dim = hidden_dim\n\n        self.lstm = nn.LSTM(\n            input_size=num_features,\n            hidden_size=hidden_dim,\n            num_layers=num_layers,\n            batch_first=True,\n            dropout=dropout if num_layers > 1 else 0.0\n        )\n\n        self.fc = nn.Sequential(\n            nn.Linear(hidden_dim, 64),\n            nn.ReLU(),\n            nn.Dropout(dropout),\n            nn.Linear(64, output_len)\n        )\n\n    def forward(self, x):\n        # x shape: (batch_size, seq_len, num_features)\n        out, (hn, cn) = self.lstm(x)\n        \n        # Take the last timestep\'s hidden state\n        last_hidden = out[:, -1, :]  # shape: (batch_size, hidden_dim)\n        \n        # Project to future forecast horizon\n        preds = self.fc(last_hidden)  # shape: (batch_size, output_len)\n        return preds\n\nif __name__ == "__main__":\n    # Sanity check\n    model = LSTMForecaster(num_features=14, hidden_dim=64, num_layers=2, output_len=24)\n    dummy_input = torch.randn(32, 168, 14)\n    output = model(dummy_input)\n    print("LSTM Output shape:", output.shape)  # Expected: (32, 24)\n    print(f"Total trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")\n', 'src/models/mlp.py': 'import torch\nimport torch.nn as nn\n\nclass MLPForecaster(nn.Module):\n    """\n    Classical Multi-Layer Perceptron (MLP) baseline.\n    Flattens the temporal input window (seq_len * num_features) and maps to output_len.\n    """\n    def __init__(\n        self,\n        seq_len: int = 168,\n        num_features: int = 14,\n        hidden_dims: list = [256, 128, 64],\n        output_len: int = 24,\n        dropout: float = 0.2\n    ):\n        super(MLPForecaster, self).__init__()\n        input_dim = seq_len * num_features\n\n        layers = []\n        prev_dim = input_dim\n        for h_dim in hidden_dims:\n            layers.append(nn.Linear(prev_dim, h_dim))\n            layers.append(nn.ReLU())\n            layers.append(nn.Dropout(dropout))\n            prev_dim = h_dim\n        layers.append(nn.Linear(prev_dim, output_len))\n\n        self.network = nn.Sequential(*layers)\n\n    def forward(self, x):\n        # x shape: (batch_size, seq_len, num_features)\n        x_flat = x.reshape(x.size(0), -1)\n        return self.network(x_flat)\n\nif __name__ == "__main__":\n    model = MLPForecaster()\n    dummy = torch.randn(32, 168, 14)\n    out = model(dummy)\n    print("MLP Output shape:", out.shape)\n    print(f"Total trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")\n', 'src/phase2.py': '"""Reproducible Phase 2 experiment; preserves original saved artifacts."""\nimport argparse\nimport os\nimport hashlib\nimport json\nimport random\nimport sys\nimport time\nfrom pathlib import Path\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.preprocessing import MinMaxScaler\nfrom torch.utils.data import DataLoader, Dataset\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT))\nfrom src.train import get_model, train_epoch, evaluate_epoch, predict_all\nfrom src.evaluate import calculate_metrics, calculate_hourly_mae, plot_sample_forecast, plot_horizon_error\nFEATURES = [\'Global_active_power\', \'Global_reactive_power\', \'Voltage\', \'Global_intensity\',\n            \'Sub_metering_1\', \'Sub_metering_2\', \'Sub_metering_3\', \'hour_sin\', \'hour_cos\',\n            \'day_sin\', \'day_cos\', \'month_sin\', \'month_cos\', \'is_weekend\']\n\ndef write_json(path, value):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2, default=str), encoding=\'utf-8\')\n\ndef prepare(out):\n    raw = ROOT / \'data/raw/household_power_consumption.txt\'\n    if not raw.exists():\n        from src.download_and_preprocess import download_and_extract\n        download_and_extract()\n    with raw.open(\'rb\') as handle:\n        raw_hash = hashlib.file_digest(handle, \'sha256\').hexdigest()\n    df = pd.read_csv(raw, sep=\';\', na_values=[\'?\', \'\'], low_memory=False)\n    idx = pd.to_datetime(df.pop(\'Date\') + \' \' + df.pop(\'Time\'), format=\'%d/%m/%Y %H:%M:%S\')\n    df.index = pd.DatetimeIndex(idx, name=\'datetime\')\n    df = df.apply(pd.to_numeric, errors=\'coerce\').sort_index()\n    if not df.index.is_unique:\n        raise ValueError(\'Duplicate raw timestamps\')\n    observed = df[\'Global_active_power\'].notna().resample(\'1h\').sum()\n    hourly = df.resample(\'1h\').mean().ffill().dropna()\n    hourly[\'target_observed\'] = observed.reindex(hourly.index).ge(60)\n    for prefix, values, period in [(\'hour\', hourly.index.hour, 24), (\'day\', hourly.index.dayofweek, 7), (\'month\', hourly.index.month-1, 12)]:\n        hourly[prefix+\'_sin\'] = np.sin(2*np.pi*values/period)\n        hourly[prefix+\'_cos\'] = np.cos(2*np.pi*values/period)\n    hourly[\'is_weekend\'] = (hourly.index.dayofweek >= 5).astype(int)\n    if not hourly.index.to_series().diff().dropna().eq(pd.Timedelta(hours=1)).all():\n        raise ValueError(\'Irregular hourly timestamps\')\n    audit = {\'raw_sha256\': raw_hash, \'raw_rows\': len(df), \'hourly_rows\': len(hourly),\n             \'start\': str(hourly.index[0]), \'end\': str(hourly.index[-1]),\n             \'missing_raw_by_column\': df.isna().sum().to_dict(),\n             \'fully_observed_target_hours\': int(hourly.target_observed.sum()),\n             \'excluded_incomplete_target_hours\': int((~hourly.target_observed).sum()),\n             \'preprocessing\': \'Observed hourly means, causal forward fill; labels require 60 observed minutes.\', \'features\': FEATURES}\n    hourly.to_csv(out/\'hourly_causal.csv\')\n    write_json(out/\'data_audit.json\', audit)\n    return hourly, audit\n\nclass Windows(Dataset):\n    def __init__(self, x, y, observed, input_len=168, output_len=24):\n        self.x = np.asarray(x, dtype=np.float32)\n        self.y = np.asarray(y, dtype=np.float32).reshape(-1)\n        observed = np.asarray(observed, dtype=bool)\n        if input_len < 1 or output_len < 1:\n            raise ValueError(\'Window lengths must be positive\')\n        if self.x.ndim != 2 or len(self.x) != len(self.y) or observed.shape != self.y.shape:\n            raise ValueError(\'Features, targets and observation mask must have matching rows\')\n        if not np.isfinite(self.x).all() or not np.isfinite(self.y).all():\n            raise ValueError(\'Window data must be finite\')\n        self.input_len, self.output_len = input_len, output_len\n        self.starts = np.array([i for i in range(len(x)-input_len-output_len+1)\n                                if observed[i+input_len:i+input_len+output_len].all()], dtype=int)\n        if not len(self.starts):\n            raise ValueError(\'No valid windows\')\n    def __len__(self):\n        return len(self.starts)\n    def __getitem__(self, i):\n        s = self.starts[i]\n        return torch.from_numpy(self.x[s:s+self.input_len]), torch.from_numpy(self.y[s+self.input_len:s+self.input_len+self.output_len])\n\ndef split_data(df, batch_size=128, seed=42):\n    n = len(df)\n    bounds = [0, int(n*.70), int(n*.85), n]\n    fs, ts = MinMaxScaler(), MinMaxScaler()\n    fs.fit(df.iloc[:bounds[1]][FEATURES]); ts.fit(df.iloc[:bounds[1]][[\'Global_active_power\']])\n    datasets, loaders, info = [], [], {}\n    for i, name in enumerate([\'train\', \'validation\', \'test\']):\n        part = df.iloc[bounds[i]:bounds[i+1]]\n        ds = Windows(fs.transform(part[FEATURES]), ts.transform(part[[\'Global_active_power\']]), part.target_observed.to_numpy(dtype=bool))\n        datasets.append(ds)\n        loaders.append(DataLoader(ds, batch_size=batch_size, shuffle=i==0, generator=torch.Generator().manual_seed(seed)))\n        info[name] = {\'rows\':len(part), \'windows\':len(ds), \'start\':str(part.index[0]), \'end\':str(part.index[-1]),\n                      \'first_forecast\':str(part.index[ds.starts[0]+168]), \'last_forecast_end\':str(part.index[ds.starts[-1]+191])}\n    info[\'boundary_indices\'] = bounds\n    info[\'scalers\'] = {\'feature_min\':fs.data_min_.tolist(), \'feature_max\':fs.data_max_.tolist(),\n                       \'target_min\':ts.data_min_.tolist(), \'target_max\':ts.data_max_.tolist()}\n    return datasets, loaders, fs, ts, info\n\ndef inverse(a, scaler):\n    return scaler.inverse_transform(a.reshape(-1, 1)).reshape(a.shape)\n\ndef train_model(name, loaders, scaler, args, models, images):\n    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)\n    loaders[0].generator.manual_seed(args.seed)\n    model = get_model(name, 14, 168, 24).to(args.device)\n    opt = torch.optim.Adam(model.parameters(), lr=args.lr)\n    loss = torch.nn.MSELoss()\n    best, stale, records = float(\'inf\'), 0, []\n    start = time.perf_counter()\n    for epoch in range(1, args.epochs+1):\n        t = time.perf_counter()\n        tr = train_epoch(model, loaders[0], loss, opt, args.device)\n        va = evaluate_epoch(model, loaders[1], loss, args.device)\n        records.append({\'epoch\':epoch, \'train_mse_scaled\':tr, \'validation_mse_scaled\':va, \'seconds\':time.perf_counter()-t})\n        print(f\'{name} epoch {epoch}: train={tr:.6f} val={va:.6f} time={records[-1]["seconds"]:.1f}s\', flush=True)\n        if va < best:\n            best, stale = va, 0\n            torch.save(model.state_dict(), models/f\'best_{name}.pt\')\n        else:\n            stale += 1\n        pd.DataFrame(records).to_csv(models/f\'history_{name}.csv\', index=False)\n        if stale >= args.patience: break\n    elapsed = time.perf_counter()-start\n    model.load_state_dict(torch.load(models/f\'best_{name}.pt\', weights_only=True, map_location=\'cpu\'))\n    p, y = predict_all(model, loaders[2], args.device)\n    p, y = inverse(p, scaler), inverse(y, scaler)\n    np.save(models/f\'preds_{name}.npy\', p); np.save(models/\'trues_test.npy\', y)\n    plt.figure(figsize=(7, 3.3))\n    plt.plot([r[\'epoch\'] for r in records], [r[\'train_mse_scaled\'] for r in records], label=\'Training\')\n    plt.plot([r[\'epoch\'] for r in records], [r[\'validation_mse_scaled\'] for r in records], label=\'Validation\')\n    plt.xlabel(\'Epoch\'); plt.ylabel(\'MSE (scaled)\'); plt.title(name.upper()+\' learning curve\'); plt.legend(); plt.tight_layout()\n    plt.savefig(images/f\'{name}_loss_curve.png\', dpi=220); plt.close()\n    result = {\'model\':name, **calculate_metrics(y,p), \'parameters\':sum(p.numel() for p in model.parameters()),\n              \'epochs_run\':len(records), \'best_epoch\':int(np.argmin([r[\'validation_mse_scaled\'] for r in records])+1),\n              \'best_validation_mse\':best, \'training_seconds\':elapsed}\n    write_json(models/f\'metrics_{name}.json\', result)\n    return p, y, result\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--epochs\', type=int, default=12)\n    ap.add_argument(\'--batch-size\', type=int, default=128)\n    ap.add_argument(\'--patience\', type=int, default=4)\n    ap.add_argument(\'--lr\', type=float, default=.001)\n    ap.add_argument(\'--seed\', type=int, default=42)\n    ap.add_argument(\'--threads\', type=int, default=4)\n    ap.add_argument(\'--device\', choices=[\'auto\', \'cpu\', \'cuda\'], default=\'auto\')\n    ap.add_argument(\'--output\', type=Path, default=ROOT/\'submission/phase2\')\n    args = ap.parse_args()\n    if min(args.epochs, args.batch_size, args.patience, args.threads) < 1: ap.error(\'Positive counts required\')\n    if not np.isfinite(args.lr) or args.lr <= 0: ap.error(\'Learning rate must be finite and positive\')\n    if args.device == \'auto\': args.device = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    if args.device == \'cuda\' and not torch.cuda.is_available(): ap.error(\'CUDA is unavailable\')\n    os.environ.setdefault(\'CUBLAS_WORKSPACE_CONFIG\', \':4096:8\')\n    torch.backends.cudnn.benchmark = False\n    torch.set_num_threads(args.threads); torch.use_deterministic_algorithms(True)\n    out = args.output.resolve()\n    results, images, models = [out/p for p in [\'results\',\'images\',\'models\']]\n    for p in [results,images,models]: p.mkdir(parents=True, exist_ok=True)\n    write_json(results/\'run_config.json\', {**vars(args), \'torch_version\':torch.__version__, \'numpy_version\':np.__version__,\n               \'pandas_version\':pd.__version__, \'python\':sys.version, \'device\':args.device, \'input_hours\':168,\n               \'forecast_hours\':24, \'created_at\':pd.Timestamp.now().isoformat()})\n    df, audit = prepare(results)\n    ds, loaders, fs, ts, info = split_data(df, args.batch_size, args.seed)\n    write_json(results/\'split_manifest.json\', info)\n    predictions, rows = {}, []\n    for name in [\'mlp\',\'cnn1d\',\'lstm\']:\n        p, y, row = train_model(name, loaders, ts, args, models, images)\n        predictions[name.upper()] = p; rows.append(row)\n        pd.DataFrame(rows).to_csv(results/\'model_comparison.csv\', index=False)\n    for name, lag in [(\'Persistence\',1),(\'Seasonal 24h\',24),(\'Seasonal 168h\',168)]:\n        base = np.stack([np.repeat(ds[2].y[s+167],24) if lag==1 else ds[2].y[s+168-lag:s+192-lag] for s in ds[2].starts])\n        base = inverse(base,ts); predictions[name] = base\n        np.save(models/(\'preds_\'+name.lower().replace(\' \',\'_\')+\'.npy\'),base)\n        rows.append({\'model\':name, **calculate_metrics(y,base), \'parameters\':0, \'epochs_run\':0, \'best_epoch\':0, \'training_seconds\':0})\n    pd.DataFrame(rows).to_csv(results/\'model_comparison.csv\', index=False)\n    h = {name:calculate_hourly_mae(y,p) for name,p in predictions.items()}\n    pd.DataFrame(h, index=pd.Index(range(1,25),name=\'hour_ahead\')).to_csv(results/\'horizon_mae.csv\')\n    plot_horizon_error(h,images/\'horizon_mae_error.png\')\n    plot_sample_forecast(y,{k:v for k,v in predictions.items() if k in [\'MLP\',\'CNN1D\',\'LSTM\',\'Seasonal 24h\']},[0,len(y)//2,len(y)-1],images/\'forecast_comparison_24h.png\')\n    origins = df.index[info[\'boundary_indices\'][2]+ds[2].starts+168]\n    pd.DataFrame({\'forecast_start\':origins}).to_csv(results/\'test_forecast_origins.csv\',index=False)\n    long = {\'forecast_start\':np.repeat(origins.astype(str),24), \'hour_ahead\':np.tile(np.arange(1,25),len(y)), \'actual_kw\':y.ravel()}\n    for name,p in predictions.items(): long[name+\'_kw\'] = p.ravel()\n    pd.DataFrame(long).to_csv(results/\'test_predictions.csv.gz\',index=False,compression=\'gzip\')\n    best_name = min([\'MLP\',\'CNN1D\',\'LSTM\'],key=lambda k:calculate_metrics(y,predictions[k])[\'MAE (kW)\'])\n    errors = np.abs(y-predictions[best_name]).mean(axis=1); worst = int(errors.argmax())\n    plot_sample_forecast(y,{best_name:predictions[best_name]},[worst],images/\'worst_forecast.png\')\n    threshold = float(df.iloc[:info[\'boundary_indices\'][1]].Global_active_power.quantile(.9)); peak = y>=threshold\n    write_json(results/\'error_analysis.json\', {\'best_neural_by_test_mae\':best_name,\'worst_sample_index\':worst,\n               \'worst_sample_start\':str(origins[worst]),\'worst_sample_mae_kw\':float(errors[worst]),\n               \'train_90th_percentile_kw\':threshold,\'peak_target_count\':int(peak.sum()),\n               \'peak_mae_kw\':float(np.abs(y-predictions[best_name])[peak].mean()),\n               \'nonpeak_mae_kw\':float(np.abs(y-predictions[best_name])[~peak].mean()),\n               \'note\':\'Overlapping forecast windows are not independent. This file describes one seed; use all_runs.csv for repeated-run analysis.\'})\n    write_json(results/\'completion.json\',{\'complete\':True,\'models\':list(predictions),\'test_windows\':len(y)})\n    print(pd.DataFrame(rows).to_string(index=False),flush=True)\n\nif __name__ == \'__main__\': main()\n', 'src/predict.py': '"""Forecast the next 24 hours from a prepared hourly CSV and a Phase 2 checkpoint."""\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT))\nfrom src.phase2 import FEATURES\nfrom src.train import get_model\n\ndef forecast(csv, model_name, artifacts):\n    df = pd.read_csv(csv, index_col=\'datetime\', parse_dates=True)\n    if len(df) < 168 or not df.index.is_monotonic_increasing or not df.index.is_unique:\n        raise ValueError(\'Input must contain at least 168 ordered, unique hourly rows\')\n    tail = df.iloc[-168:]\n    if not tail.index.to_series().diff().dropna().eq(pd.Timedelta(hours=1)).all():\n        raise ValueError(\'Last 168 hours must be contiguous\')\n    x = tail[FEATURES].to_numpy(dtype=np.float32)\n    if not np.isfinite(x).all(): raise ValueError(\'Features must all be finite\')\n    scaling = json.loads((artifacts/\'results/split_manifest.json\').read_text())[\'scalers\']\n    low, high = np.array(scaling[\'feature_min\']), np.array(scaling[\'feature_max\'])\n    x = (x-low)/np.where(high==low, 1, high-low)\n    model = get_model(model_name,14,168,24)\n    model.load_state_dict(torch.load(artifacts/\'models\'/f\'best_{model_name}.pt\',map_location=\'cpu\',weights_only=True))\n    model.eval()\n    with torch.no_grad(): pred = model(torch.tensor(x[None],dtype=torch.float32)).numpy().ravel()\n    tmin, tmax = scaling[\'target_min\'][0], scaling[\'target_max\'][0]\n    pred = pred*(tmax-tmin if tmax!=tmin else 1)+tmin\n    return pd.DataFrame({\'datetime\':pd.date_range(tail.index[-1]+pd.Timedelta(hours=1),periods=24,freq=\'h\'),\n                         \'predicted_active_power_kw\':pred})\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--model\',choices=[\'mlp\',\'cnn1d\',\'lstm\'],default=\'mlp\')\n    p.add_argument(\'--artifacts\',type=Path,default=ROOT/\'submission/phase2\')\n    p.add_argument(\'--csv\',type=Path)\n    p.add_argument(\'--output\',type=Path)\n    a = p.parse_args(); torch.set_num_threads(2)\n    result = forecast(a.csv or a.artifacts/\'results/hourly_causal.csv\',a.model,a.artifacts)\n    if a.output:\n        a.output.parent.mkdir(parents=True,exist_ok=True); result.to_csv(a.output,index=False)\n    print(result.to_string(index=False))\n', 'src/train.py': 'import argparse\nimport time\nimport os\nimport torch\nimport torch.nn as nn\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport sys\nfrom pathlib import Path\n\n# Ensure project root is in sys.path\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\nif str(PROJECT_ROOT) not in sys.path:\n    sys.path.insert(0, str(PROJECT_ROOT))\n\nfrom src.data_loader import load_and_split_data\nfrom src.models.mlp import MLPForecaster\nfrom src.models.cnn1d import CNN1DForecaster\nfrom src.models.lstm import LSTMForecaster\nfrom src.evaluate import calculate_metrics\n\ndef get_model(model_name: str, num_features: int, input_len: int, output_len: int):\n    model_name = model_name.lower()\n    if model_name == "mlp":\n        return MLPForecaster(seq_len=input_len, num_features=num_features, output_len=output_len)\n    elif model_name in ["cnn", "cnn1d", "1d-cnn"]:\n        return CNN1DForecaster(num_features=num_features, output_len=output_len)\n    elif model_name == "lstm":\n        return LSTMForecaster(num_features=num_features, hidden_dim=64, num_layers=2, output_len=output_len)\n    else:\n        raise ValueError(f"Unknown model name: {model_name}. Choose from: mlp, cnn1d, lstm")\n\ndef train_epoch(model, loader, criterion, optimizer, device):\n    model.train()\n    total_loss = 0.0\n    for X_batch, y_batch in loader:\n        X_batch, y_batch = X_batch.to(device), y_batch.to(device)\n        optimizer.zero_grad()\n        preds = model(X_batch)\n        loss = criterion(preds, y_batch)\n        loss.backward()\n        optimizer.step()\n        total_loss += loss.item() * len(X_batch)\n    return total_loss / len(loader.dataset)\n\ndef evaluate_epoch(model, loader, criterion, device):\n    model.eval()\n    total_loss = 0.0\n    with torch.no_grad():\n        for X_batch, y_batch in loader:\n            X_batch, y_batch = X_batch.to(device), y_batch.to(device)\n            preds = model(X_batch)\n            loss = criterion(preds, y_batch)\n            total_loss += loss.item() * len(X_batch)\n    return total_loss / len(loader.dataset)\n\ndef predict_all(model, loader, device):\n    model.eval()\n    all_preds, all_trues = [], []\n    with torch.no_grad():\n        for X_batch, y_batch in loader:\n            X_batch = X_batch.to(device)\n            preds = model(X_batch).cpu().numpy()\n            all_preds.append(preds)\n            all_trues.append(y_batch.numpy())\n    return np.vstack(all_preds), np.vstack(all_trues)\n\ndef main():\n    parser = argparse.ArgumentParser(description="Train time-series forecasting models.")\n    parser.add_argument("--model", type=str, default="lstm", choices=["mlp", "cnn1d", "lstm"], help="Model to train")\n    parser.add_argument("--epochs", type=int, default=15, help="Number of training epochs")\n    parser.add_argument("--batch_size", type=int, default=64, help="Batch size")\n    parser.add_argument("--lr", type=float, default=0.001, help="Learning rate")\n    parser.add_argument("--patience", type=int, default=4, help="Early stopping patience")\n    args = parser.parse_args()\n\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    print(f"Using device: {device}")\n    print(f"Training Model: {args.model.upper()}")\n\n    # 1. Load Data\n    print("\\nLoading dataset and creating sliding windows...")\n    train_loader, val_loader, test_loader, target_scaler, meta = load_and_split_data(\n        input_len=168, output_len=24, batch_size=args.batch_size\n    )\n\n    # 2. Initialize Model\n    model = get_model(args.model, meta["num_features"], meta["input_len"], meta["output_len"]).to(device)\n    criterion = nn.MSELoss()\n    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)\n\n    # 3. Training Loop with Early Stopping\n    save_dir = Path(__file__).resolve().parent.parent / "saved_models"\n    save_dir.mkdir(parents=True, exist_ok=True)\n    best_model_path = save_dir / f"best_{args.model}.pt"\n\n    best_val_loss = float("inf")\n    patience_counter = 0\n    train_losses, val_losses = [], []\n\n    print(f"\\nStarting training for {args.epochs} epochs...")\n    start_time = time.time()\n    for epoch in range(1, args.epochs + 1):\n        t0 = time.time()\n        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)\n        val_loss = evaluate_epoch(model, val_loader, criterion, device)\n        elapsed = time.time() - t0\n\n        train_losses.append(train_loss)\n        val_losses.append(val_loss)\n\n        improved = ""\n        if val_loss < best_val_loss:\n            best_val_loss = val_loss\n            torch.save(model.state_dict(), best_model_path)\n            patience_counter = 0\n            improved = " [SAVED BEST]"\n        else:\n            patience_counter += 1\n\n        print(f"Epoch [{epoch:02d}/{args.epochs:02d}] - Train Loss: {train_loss:.5f} | Val Loss: {val_loss:.5f} | Time: {elapsed:.1f}s{improved}")\n\n        if patience_counter >= args.patience:\n            print(f"Early stopping triggered at epoch {epoch}!")\n            break\n\n    total_time = time.time() - start_time\n    print(f"\\nTraining completed in {total_time:.1f} seconds.")\n\n    # 4. Evaluation on Test Set\n    print("\\nLoading best model for test set evaluation...")\n    model.load_state_dict(torch.load(best_model_path))\n\n    scaled_preds, scaled_trues = predict_all(model, test_loader, device)\n\n    # Inverse transform to original kW scale\n    # target_scaler was fitted on 2D shape (N, 1), so reshape before inverse transform\n    unscaled_preds = target_scaler.inverse_transform(scaled_preds.reshape(-1, 1)).reshape(scaled_preds.shape)\n    unscaled_trues = target_scaler.inverse_transform(scaled_trues.reshape(-1, 1)).reshape(scaled_trues.shape)\n\n    metrics = calculate_metrics(unscaled_trues, unscaled_preds)\n    print("\\n" + "=" * 45)\n    print(f"     TEST SET EVALUATION RESULTS ({args.model.upper()})")\n    print("=" * 45)\n    for k, v in metrics.items():\n        print(f"  {k:15s} : {v:.4f}")\n    print("=" * 45)\n\n    # Save predictions array for comparison\n    np.save(save_dir / f"preds_{args.model}.npy", unscaled_preds)\n    np.save(save_dir / "trues_test.npy", unscaled_trues)\n\n    # Plot & Save Loss Curve\n    plots_dir = Path(__file__).resolve().parent.parent / "plots"\n    plots_dir.mkdir(parents=True, exist_ok=True)\n    plt.figure(figsize=(8, 4.5))\n    plt.plot(train_losses, label="Train Loss (MSE)", linewidth=2)\n    plt.plot(val_losses, label="Validation Loss (MSE)", linewidth=2)\n    plt.title(f"{args.model.upper()} - Training & Validation Loss", fontweight="bold")\n    plt.xlabel("Epoch")\n    plt.ylabel("MSE Loss (Scaled)")\n    plt.grid(True, linestyle=":", alpha=0.6)\n    plt.legend()\n    plt.tight_layout()\n    loss_plot_path = plots_dir / f"{args.model}_loss_curve.png"\n    plt.savefig(loss_plot_path, dpi=300)\n    plt.close()\n    print(f"Saved loss curve plot to: {loss_plot_path}")\n\nif __name__ == "__main__":\n    main()\n', 'tests/test_phase2.py': '"""Scientific protocol checks, runnable with unittest (no network required)."""\nimport sys\nimport unittest\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom src.phase2 import FEATURES, Windows, split_data\nfrom src.train import get_model\nfrom src.evaluate import calculate_metrics\n\nclass ProtocolTests(unittest.TestCase):\n    def test_windows_contain_past_inputs_and_future_labels(self):\n        x = np.arange(220, dtype=np.float32)\n        ds = Windows(x[:, None], x, np.ones(220, dtype=bool))\n        self.assertEqual(len(ds), 29)\n        features, target = ds[0]\n        np.testing.assert_array_equal(features[:, 0], np.arange(168))\n        np.testing.assert_array_equal(target, np.arange(168, 192))\n\n    def test_incomplete_labels_are_excluded(self):\n        x = np.arange(220, dtype=np.float32)\n        observed = np.ones(220, dtype=bool)\n        observed[180] = False\n        ds = Windows(x[:, None], x, observed)\n        self.assertTrue(all(not (s+168 <= 180 < s+192) for s in ds.starts))\n\n    def test_scaler_fits_training_only_and_splits_are_disjoint(self):\n        df = pd.DataFrame(np.arange(2000)[:, None] * np.ones((1, 14)),\n                          index=pd.date_range(\'2020-01-01\', periods=2000, freq=\'h\'), columns=FEATURES)\n        df[\'target_observed\'] = True\n        datasets, _, fs, _, info = split_data(df)\n        self.assertEqual(fs.data_max_[0], 1399)\n        self.assertGreater(datasets[-1].x.max(), 1)\n        self.assertLess(info[\'train\'][\'end\'], info[\'validation\'][\'start\'])\n        self.assertLess(info[\'validation\'][\'end\'], info[\'test\'][\'start\'])\n\n    def test_models_produce_24_outputs_and_receive_gradients(self):\n        torch.set_num_threads(2)\n        for name in [\'mlp\', \'cnn1d\', \'lstm\']:\n            model = get_model(name, 14, 168, 24)\n            pred = model(torch.randn(2, 168, 14))\n            self.assertEqual(tuple(pred.shape), (2, 24))\n            pred.square().mean().backward()\n            self.assertTrue(all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters()))\n\n    def test_metrics_known_answer(self):\n        y = np.array([[1., 2.], [3., 4.]])\n        perfect = calculate_metrics(y, y)\n        self.assertEqual(perfect[\'MAE (kW)\'], 0)\n        self.assertEqual(perfect[\'R2 Score\'], 1)\n        shifted = calculate_metrics(y, y+1)\n        self.assertEqual(shifted[\'MAE (kW)\'], 1)\n        self.assertEqual(shifted[\'RMSE (kW)\'], 1)\n\nif __name__ == \'__main__\':\n    unittest.main(verbosity=2)\n', 'tests/test_predict.py': '"""Check that saved scaling and checkpoints reproduce inference in kW."""\nimport json\nimport tempfile\nimport unittest\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom src.phase2 import FEATURES, Windows\nfrom src.predict import forecast\nfrom src.train import get_model\n\n\nclass InferenceTests(unittest.TestCase):\n    def test_mlp_accepts_noncontiguous_inputs(self):\n        model = get_model(\'mlp\', 14, 168, 24).eval()\n        x = torch.randn(2, 14, 168).transpose(1, 2)\n        self.assertFalse(x.is_contiguous())\n        with torch.no_grad():\n            torch.testing.assert_close(model(x), model(x.contiguous()))\n\n    def test_checkpoint_and_scaling_round_trip(self):\n        with tempfile.TemporaryDirectory() as folder:\n            root = Path(folder)\n            (root / \'models\').mkdir()\n            (root / \'results\').mkdir()\n            model = get_model(\'mlp\', 14, 168, 24)\n            for param in model.parameters():\n                torch.nn.init.zeros_(param)\n            torch.save(model.state_dict(), root / \'models/best_mlp.pt\')\n            scale = {\'feature_min\': [0.] * 14, \'feature_max\': [1.] * 14,\n                     \'target_min\': [2.], \'target_max\': [6.]}\n            (root / \'results/split_manifest.json\').write_text(json.dumps({\'scalers\': scale}))\n            df = pd.DataFrame(np.ones((168, 14)), columns=FEATURES,\n                              index=pd.date_range(\'2020-01-01\', periods=168, freq=\'h\', name=\'datetime\'))\n            csv = root / \'input.csv\'\n            df.to_csv(csv)\n            result = forecast(csv, \'mlp\', root)\n            self.assertEqual(len(result), 24)\n            self.assertEqual(result.datetime.iloc[0], df.index[-1] + pd.Timedelta(hours=1))\n            np.testing.assert_allclose(result.predicted_active_power_kw, 2.)\n            df.index = df.index.where(df.index != df.index[-1], df.index[-1] + pd.Timedelta(hours=1))\n            df.to_csv(csv)\n            with self.assertRaisesRegex(ValueError, \'contiguous\'):\n                forecast(csv, \'mlp\', root)\n\n    def test_reject_misaligned_and_nonfinite_windows(self):\n        with self.assertRaisesRegex(ValueError, \'matching rows\'):\n            Windows(np.zeros((200, 2)), np.zeros(199), np.ones(200))\n        with self.assertRaisesRegex(ValueError, \'finite\'):\n            Windows(np.full((200, 2), np.nan), np.zeros(200), np.ones(200))\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n'}
for name, content in sources.items():
    p = ROOT / name
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding="utf-8")
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)


## Training and evaluation
A 168-hour history predicts 24 future hourly means. Scalers use training rows only; all target hours must have 60 observed readings. Validation MSE selects checkpoints. Test scores are reported without test-based tuning.


In [ ]:
for seed in [42, 43, 44]:
    subprocess.run([sys.executable, "src/phase2.py", "--device", "cuda", "--epochs", "30", "--patience", "6", "--batch-size", "128", "--seed", str(seed), "--output", f"artifacts/seed_{seed}"], check=True)


In [ ]:
import pandas as pd
frames = []
for seed in [42, 43, 44]:
    frame = pd.read_csv(f"artifacts/seed_{seed}/results/model_comparison.csv")
    frame["seed"] = seed
    frames.append(frame)
scores = pd.concat(frames, ignore_index=True)
scores.to_csv("artifacts/all_runs.csv", index=False)
summary = scores.groupby("model")[["MAE (kW)", "RMSE (kW)", "MAPE (%)", "R2 Score"]].agg(["mean", "std"])
summary.to_csv("artifacts/summary.csv")
display(summary)
subprocess.run([sys.executable, "src/predict.py", "--artifacts", "artifacts/seed_42", "--output", "artifacts/example_forecast.csv"], check=True)
import shutil
shutil.make_archive("/kaggle/working/forecast_results", "zip", ROOT / "artifacts")
print("Artifacts saved to /kaggle/working/forecast_results.zip")
